# 05. 2024→2025년 20~50대 인구 변화량 생성

고양시·파주시 250m SGIS 격자 인구자료에서 20·30·40·50대 인구를 합산하고, 공급구조 유지 132개 격자에 공간 매칭한다. 최종 변수는 `2025년 20~50대 인구 − 2024년 20~50대 인구`이며, 회귀 해석 편의를 위해 100명 단위 변수도 함께 저장한다.

In [1]:
import sys
from pathlib import Path
import unicodedata

import geopandas as gpd
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))
from config import *

setup_notebook()
print('2024년 원자료:', SGIS_DIR_2024)
print('2025년 원자료:', SGIS_DIR_2025)
print('저장 파일:', POPULATION_CHANGE_PATH)

2024년 원자료: C:\Users\금경훈\Desktop\Gachon\3-2\UROP\github_2026-UROP\gtx_a_bus_analysis\data\raw\sgis_2024
2025년 원자료: C:\Users\금경훈\Desktop\Gachon\3-2\UROP\github_2026-UROP\gtx_a_bus_analysis\data\raw\sgis_2025
저장 파일: C:\Users\금경훈\Desktop\Gachon\3-2\UROP\github_2026-UROP\gtx_a_bus_analysis\data\processed\11_20~50대인구변화_2024_2025_250m격자.csv


In [2]:
def nfc(value):
    return unicodedata.normalize('NFC', str(value))


def find_age_shp(sgis_dir, city_folder, age, year):
    city_dir = sgis_dir / city_folder
    matches = [
        folder / 'vl_blk.shp'
        for folder in city_dir.iterdir()
        if folder.is_dir()
        and nfc(f'{age}대 인구 수') in nfc(folder.name)
        and str(year) + '10' in folder.name
        and (folder / 'vl_blk.shp').exists()
    ]
    assert len(matches) == 1, f'{year}년 {city_folder} {age}대: {matches}'
    return matches[0]


def match_grid_value(centers, shp_path):
    population_grid = gpd.read_file(shp_path)[['val', 'geometry']].to_crs('EPSG:5179')
    population_grid['val'] = pd.to_numeric(population_grid['val'], errors='coerce')

    joined = (
        gpd.sjoin(centers[['셀 ID', 'geometry']], population_grid, how='left', predicate='within')
        .drop_duplicates('셀 ID')
    )

    # 중심점이 인구격자 경계에 놓인 경우 intersects로 한 번 더 매칭한다.
    missing_ids = joined.loc[joined['val'].isna(), '셀 ID']
    if len(missing_ids):
        fallback = gpd.sjoin(
            centers[centers['셀 ID'].isin(missing_ids)][['셀 ID', 'geometry']],
            population_grid, how='left', predicate='intersects'
        ).drop_duplicates('셀 ID')
        joined.loc[joined['셀 ID'].isin(missing_ids), 'val'] = (
            joined.loc[joined['셀 ID'].isin(missing_ids), '셀 ID']
            .map(fallback.set_index('셀 ID')['val'])
        )
    return joined.set_index('셀 ID')['val']


def build_working_age_population(centers, sgis_dir, year):
    result = centers[['셀 ID', '중심점 위도', '중심점 경도']].copy()
    age_columns = []
    for age in (20, 30, 40, 50):
        values = []
        for city_folder in ('고양인구', '파주인구'):
            values.append(match_grid_value(centers, find_age_shp(sgis_dir, city_folder, age, year)))
        column = f'{year}년 {age}대 인구'
        result[column] = result['셀 ID'].map(pd.concat(values, axis=1).sum(axis=1, min_count=1))
        age_columns.append(column)

    result[f'{year}년 인구 원자료 결측 여부'] = result[age_columns].isna().any(axis=1)
    result[age_columns] = result[age_columns].fillna(0).clip(lower=0)
    result[f'{year}년 20~50대 인구'] = result[age_columns].sum(axis=1)
    return result.sort_values('셀 ID').reset_index(drop=True)

In [3]:
centers = stable_centers()
population_2024 = build_working_age_population(centers, SGIS_DIR_2024, 2024)
population_2025 = build_working_age_population(centers, SGIS_DIR_2025, 2025)

population_change = population_2024.merge(
    population_2025.drop(columns=['중심점 위도', '중심점 경도']),
    on='셀 ID', how='inner', validate='one_to_one'
)
population_change['20~50대 인구 증감'] = (
    population_change['2025년 20~50대 인구'] - population_change['2024년 20~50대 인구']
)
population_change['20~50대 인구 증감(100명)'] = population_change['20~50대 인구 증감'] / 100
population_change.to_csv(POPULATION_CHANGE_PATH, index=False, encoding='utf-8-sig')

print(f'분석 격자: {len(population_change)}개')
print('2024년 결측:', int(population_change['2024년 인구 원자료 결측 여부'].sum()), '개')
print('2025년 결측:', int(population_change['2025년 인구 원자료 결측 여부'].sum()), '개')
print('저장:', POPULATION_CHANGE_PATH)
display(population_change[[
    '셀 ID', '2024년 20~50대 인구', '2025년 20~50대 인구',
    '20~50대 인구 증감', '20~50대 인구 증감(100명)'
]].head())

분석 격자: 132개
2024년 결측: 14 개
2025년 결측: 18 개
저장: C:\Users\금경훈\Desktop\Gachon\3-2\UROP\github_2026-UROP\gtx_a_bus_analysis\data\processed\11_20~50대인구변화_2024_2025_250m격자.csv


,셀 ID,2024년 20~50대 인구,2025년 20~50대 인구,20~50대 인구 증감,20~50대 인구 증감(100명)
0,44,937.0,990.0,53.0,0.53
1,76,33.0,40.0,7.0,0.07
2,559,0.0,0.0,0.0,0.00
3,619,0.0,0.0,0.0,0.00
4,765,139.0,139.0,0.0,0.00


In [4]:
summary = population_change[[
    '2024년 20~50대 인구', '2025년 20~50대 인구',
    '20~50대 인구 증감', '20~50대 인구 증감(100명)'
]].describe().round(2)
display(summary)

print('인구 증가 격자:', int((population_change['20~50대 인구 증감'] > 0).sum()), '개')
print('인구 감소 격자:', int((population_change['20~50대 인구 증감'] < 0).sum()), '개')
print('변화 없음 격자:', int((population_change['20~50대 인구 증감'] == 0).sum()), '개')

,2024년 20~50대 인구,2025년 20~50대 인구,20~50대 인구 증감,20~50대 인구 증감(100명)
count,132.00,132.00,132.00,132.00
mean,648.23,622.57,-25.67,-0.26
std,554.78,543.12,99.66,1.00
min,0.00,0.00,-599.00,-5.99
25%,101.75,78.50,-43.75,-0.44
50%,551.00,523.50,-6.00,-0.06
75%,1088.25,1067.50,4.25,0.04
max,2141.00,2098.00,248.00,2.48


인구 증가 격자: 39 개
인구 감소 격자: 73 개
변화 없음 격자: 20 개
